# 36 — Event-Driven & Serverless Architecture

**Role:** Senior AWS DevOps Engineer | **Focus:** API Gateway, EventBridge, SQS/SNS, Step Functions, Lambda

---

## Section A: Event-Driven Architecture

### Q1: EventBridge Architecture
**Question:** How does Amazon EventBridge differ from SQS/SNS? When would you use each?

**Expected Answer:**
- **SNS**: Pub/Sub fan-out. One message → many subscribers. Push-based. Simple.
- **SQS**: Queue. Decouples producer/consumer. Pull-based. Guaranteed delivery with retries.
- **EventBridge**: Event bus with content-based routing rules. Schema registry. Cross-account events. Best for complex routing.

| Use Case | Service |
|---|---|
| \"Notify 5 services when order placed\" | SNS |
| \"Buffer tasks for a worker fleet\" | SQS |
| \"Route events by type/content to different targets\" | EventBridge |
| \"React to AWS service events (EC2 state change)\" | EventBridge |

- **Pattern**: SNS → SQS fan-out (combine pub/sub with queue durability).
- **EventBridge rules**: Filter on JSON event patterns. Route `order.placed` to Lambda A, `order.cancelled` to Lambda B.

---

### Q2: SQS Deep Dive
**Question:** What's the difference between Standard and FIFO SQS queues? How do you handle Dead Letter Queues?

**Expected Answer:**
- **Standard**: At-least-once delivery, best-effort ordering. Nearly unlimited throughput.
- **FIFO**: Exactly-once processing, strict ordering. 300 msgs/sec (3000 with batching).
- **DLQ**: Messages that fail processing N times (maxReceiveCount) move to DLQ. Monitor DLQ length as a critical alert.
- **Redrive**: `StartMessageMoveTask` API to replay DLQ messages back to source queue.
- **Visibility timeout**: Must be longer than your processing time. Otherwise message reappears and gets processed twice.
- **Long polling**: Set `ReceiveWaitTimeSeconds = 20` to reduce empty API calls and cost.

---

### Q3: Step Functions Orchestration
**Question:** Design a Step Functions workflow for an order processing system: validate → charge payment → update inventory → send confirmation. How do you handle partial failures?

**Expected Answer:**
- **State types**: Task (invoke Lambda/ECS), Choice (if/else), Parallel (concurrent steps), Wait, Map (iterate).
- **Error handling**: `Catch` block per state → route to compensation logic.
- **Saga pattern**: Each step has a compensating action. If `charge payment` succeeds but `update inventory` fails, run `refund payment`.
- **Express vs. Standard**: Express for high-volume, short-duration (< 5 min). Standard for long-running, auditable.
- **Idempotency**: Each Lambda must be idempotent (use idempotency key from the event).
- **Observability**: Step Functions has built-in visual execution history. X-Ray integration for tracing.

## Section B: API Gateway

### Q4: API Gateway Patterns
**Question:** Compare REST API, HTTP API, and WebSocket API in API Gateway. When do you use each?

**Expected Answer:**
- **REST API**: Full-featured (request validation, caching, WAF, usage plans/API keys). Higher cost.
- **HTTP API**: Cheaper, faster, simpler. JWT authorizer built-in. Best for most new APIs.
- **WebSocket API**: Real-time bidirectional (chat, live dashboards, notifications).
- **Choice guide**: Use HTTP API unless you need REST-specific features (caching, request transformation, resource policies).
- **Integration types**: Lambda proxy (most common), HTTP proxy (pass-through to ALB/ECS), VPC Link (private backends).

---

### Q5: API Gateway + Lambda Throttling
**Question:** Your API Gateway-backed Lambda is getting 429 (Too Many Requests) errors during a traffic spike. How do you fix it?

**Expected Answer:**
- **Lambda concurrency**: Default 1000 per region. Request quota increase if needed.
- **Reserved concurrency**: Guarantee capacity for critical functions (but limits others).
- **Provisioned concurrency**: Pre-warm for predictable traffic.
- **API Gateway throttling**: Default 10,000 req/sec per region. Configure per-stage and per-method limits.
- **Burst**: API Gateway allows 5000 concurrent requests burst.
- **Pattern**: SQS in front of Lambda for async processing — absorbs spikes.

## Section C: Serverless DevOps

### Q6: Lambda Deployment & Versioning
**Question:** How do you deploy Lambda functions safely with rollback capability?

**Expected Answer:**
- **Versions**: Each deploy publishes a new immutable version ($LATEST is mutable).
- **Aliases**: `prod` alias points to version 42. Deploy v43, shift traffic gradually.
- **Weighted aliases**: 90% to v42, 10% to v43 (canary). Monitor errors. Shift to 100%.
- **CodeDeploy**: Automates canary/linear deployment with automatic rollback on CloudWatch alarm.
- **SAM / CDK**: Define `AutoPublishAlias` and `DeploymentPreference` in template.
- **Rollback**: Repoint alias to previous version (instant).

---

### Q7: Serverless Observability Challenges
**Question:** Observing serverless apps is different from containers. What are the unique challenges?

**Expected Answer:**
- **Ephemeral**: No persistent process to attach a monitoring agent to.
- **Cold starts**: Appear as latency spikes. Track with `Init Duration` in CloudWatch.
- **Distributed**: A single user request may invoke 5 Lambdas + 2 queues + 1 Step Function.
- **Solutions**: X-Ray/OTel for traces. Structured logging with correlation IDs. CloudWatch Lambda Insights for memory/CPU.
- **Third-party**: Datadog Lambda layer, Lumigo, or Sentry for enhanced serverless monitoring.
- **Cost**: Monitor per-function invocation count and duration. `aws lambda get-function-configuration` to check memory allocation.

## Section D: Streaming & Real-Time (Nice to Have)

### Q8: Kinesis vs. MSK (Kafka)
**Question:** You need a real-time streaming pipeline. When do you pick Kinesis over MSK (Managed Kafka)?

**Expected Answer:**
- **Kinesis Data Streams**: Fully managed, simple, good for AWS-native pipelines. Shard-based scaling.
- **MSK (Kafka)**: Better for complex event streaming, multi-consumer patterns, existing Kafka ecosystem.
- **Kinesis Firehose**: Zero-code delivery to S3, Redshift, Elasticsearch. Best for log aggregation.
- **Cost**: Kinesis charges per shard-hour + per PUT. MSK charges per broker instance.
- **Choice**: Kinesis for simple ingestion. MSK for event sourcing, CQRS, complex topologies.

---

### Q9: Pixel Streaming Infrastructure (Nice to Have)
**Question:** How would you design infrastructure for Unreal Engine Pixel Streaming on AWS?

**Expected Answer:**
- **GPU instances**: g5/g5g instances with NVIDIA GPUs. Spot for cost (if stateless sessions).
- **Networking**: Low-latency — use Global Accelerator or CloudFront for WebRTC signaling.
- **Scaling**: One Unreal instance per user session. Scale GPU fleet based on concurrent users.
- **Signaling server**: Manages WebRTC peer connections. Run on ECS/EKS.
- **TURN/STUN**: For NAT traversal. Deploy coturn or use Twilio TURN.
- **Cost optimization**: Shutdown idle sessions aggressively. Use placement groups for GPU locality.